In [ ]:
import csv
import pandas as pd
import numpy as np

In [ ]:
file = open(r"iris.csv", 'r') 
file

In [ ]:
data = csv.reader(file)  # cav.reader(df): tách các phần từ theo cột của từng hàng 
data = list(data)
data[0], data[1]

In [ ]:
data = np.array(data)
data

In [ ]:
data.shape

In [ ]:
# np.delete(arr, obj, axis=None)
data = np.delete(data, 0, 0) # del hàng 0 
data = np.delete(data, 0, 1)
print(data)
file.close # close tránh lãng phí tài nguyên file

In [ ]:
trainingSet = data[:149] # Huấn luyện 149 dong đầu
testingSet = data[149:] # dòng này để test
float(trainingSet[1][0]) + 1

In [ ]:
def compute_distance(datapoint1, datapoint2):
    result = 0
    for i in range(4):
        result += (float(datapoint1[i]) - float(datapoint2[i]))**2
    
    return result

In [ ]:
def computereKnearesNeighbor(trainingSet, item, k):
    distance = []
    for dataPoint in trainingSet:
        distance.append( 
            {
                "labels" : dataPoint[-1],
                "values" : compute_distance(dataPoint, item)
            }
        )
    
    # Sort top K nearest
    # list.sort(key=None, reverse=False) trong đó key là xác định tiêu chí sắp xếp
    distance.sort(key=lambda x: x['values'])
    # Filter top K labels
    # [biểu_thức for biến in iterable]
    labels = [item['labels'] for item in distance]
    return labels[:k]

In [ ]:
def voteTheDistances(knn):
    # set in python: ko có thứ tự, ko chứa phần tử trùng lặp, có thể modify, không chứa list and dict
    labels = set(knn)
    mx=0
    # Tìm labels có chứa số lượng nhiều nhất
    for item_label in labels:
        cnt = knn.count(item_label)
        if mx < cnt:
            mx = cnt
            result = item_label
    return result

In [ ]:
K = 5
for item in testingSet:
    knn = computereKnearesNeighbor(trainingSet, item, K)
    result = voteTheDistances(knn)
    print("GT = ", item[-1], ", Prediction =", result)

-Làm thế nào để chọn K in K-NN <br>
-Sau đây là cách dùng thư viện sklearn cho KNN

In [ ]:

df = pd.read_csv("iris.csv")
df

In [ ]:
X = df.iloc[:, 1:-1]
y = df.iloc[:, 5]

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=4
)

In [ ]:
# Chuẩn hóa data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
classifier = KNeighborsClassifier(n_neighbors=5) # n_neighbors is K
classifier.fit(X_train, y_train) # Sắp xếp lại dữ liệu theo kiểu dession tree giúp giảm access time và tránh 
                                # Lãng phí tài nguyên

In [ ]:
y_pred = classifier.predict(X_test)
y_pred != y_test

In [ ]:
# Đánh giá mô hình
from sklearn.metrics import accuracy_score
print(round(accuracy_score(y_test, y_pred), 2))

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_pred, y_test))

In [ ]:
# Vẽ biểu đồ xem nên lựa chọn K = mấy là tốt nhất
import matplotlib.pyplot as plt
error = []
for i in range(1, 30):
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train, y_train)
    y_pred_i = knn.predict(X_test)
    acy = accuracy_score(y_pred_i, y_test)

    error.append(acy)

plt.plot(range(1,30), error, color='blue', marker='o', markerfacecolor='yellow')
plt.xlabel("K value")
plt.ylabel("Accuracy")
plt.title("Accuracy vs K value")
plt.show()

==============================================================================================================
- ÔN tập KNN trên tập csv khác

In [ ]:
df = pd.read_csv("TeleCustomers.csv")
# df.drop(name_col, axis=0/1)
X = df.drop(['custcat'],axis=1)
y = df['custcat']


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

from sklearn.neighbors import KNeighborsClassifier 
K = 4
neigh = KNeighborsClassifier(n_neighbors=K).fit(X_train, y_train)
y_pred = neigh.predict(X_test)
y_pred != y_test

In [ ]:
# Metric đánh giá error_rate
from sklearn import neighbors
error_rate = []
for i in range(1, 40):
    knn = neighbors.KNeighborsClassifier(n_neighbors=i, weights='distance') # distance weight
    knn.fit(X_train, y_train)
    y_pred_i = knn.predict(X_test)
    error_rate.append(np.mean(y_pred_i != y_test)) # K cảng nhỏ, metric càng tốt

plt.plot(range(1,40), error_rate, color='blue', marker='o', markerfacecolor='red', linestyle='-')
plt.title('Error Rate vs. K Value')
plt.xlabel('K')
plt.ylabel('Error Rate')
print("Minimum error:",min(error_rate),"at K =",error_rate.index(min(error_rate)))

In [ ]:
# Metric: acccuracy
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier 
acc = []
for i in range(1, 40):
    knn = KNeighborsClassifier(n_neighbors=i, weights='distance').fit(X_train, y_train)
    y_pred_i = knn.predict(X_test)
    acc.append(accuracy_score(y_pred_i, y_test))

plt.figure(figsize=(10,6))
plt.plot(range(1,40),acc,color = 'blue',linestyle='dashed',
         marker='o',markerfacecolor='red', markersize=10)
plt.title('accuracy vs. K Value')
plt.xlabel('K')
plt.ylabel('Accuracy')
print("Maximum accuracy:",max(acc),"at K =",acc.index(max(acc)))